In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    import google.colab  # noqa: F401
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# How to make decisions from model outputs

Bradley Segal<br /> [bradley.segal@eng.ox.ac.uk](mailto:bradley.segal@eng.ox.ac.uk)<br /> Computational Health Informatics Lab, Institute of Biomedical Engineering, Department of Engineering Science, University of Oxford

Organiser: Xiao Gu, Instructor: Bradley Segal, Demonstrators: Xiao Gu and Edward Phillips.

Text is licensed under CC BY 4.0, and code under the MIT licence.

## Practical overview

This practical contains a series of short functions for you to complete, each building on the ones before, as well as a written recommendation. Fill in the code wherever you see a `...`, then run the cell. The checker under the cell shows whether your answer is correct: ✓ if it is, **Not yet** with a hint for what to change if it isn't, and **Not attempted yet** if a `...` is still in place of your code. If a cell needs an earlier task to be answered correctly, it shows **Waiting for** and the name of that task.

The practical will be marked by your demonstrators based on the work you do in the session, so please complete it in your own copy of the notebook. A progress cell at the bottom of the notebook shows which tasks you've completed. The solutions will be released after the session.

Please save a copy of this notebook to your Drive (**File > Save a copy in Drive**) and then run the setup cell at the top.

## What will you be doing?

| Part | What you do | Tasks |
|---|---|---|
| 1. How much does a policy cost? | Count a policy's outcomes and price them in dollars | 1 and 2 |
| 2. What is the optimal cutoff? | Evaluate the ROC curve, read the precision-recall curve and find each model's cheapest cutoff | 3 and 4 |
| 3. Is the money necessary? | Turn the costs into a threshold probability | 5 |
| 4. Decision curve analysis | Calculate net benefit and compare the models with the two blind choices | 6 |
| 5. Why the two answers are different | Check each model's calibration and recalibrate a score | 7 |
| 6. What would you recommend? | Recommend a policy and test it on patients held out until then | 8 |
| Extension (optional) | Find the best policy when the number of reviews is limited | A to C |

Some of these ideas will be new to you, and you don't need any prior reading to complete the practical. If you'd like to get a more detailed understanding, either during or after the session, the following are good places to start:

- [Vickers et al. 2019](https://doi.org/10.1186/s41512-019-0064-7): a step-by-step guide to interpreting a decision curve.
- [Vickers et al. 2016](https://doi.org/10.1136/bmj.i6): net benefit as a way of evaluating prediction models and diagnostic tests.
- [Van Calster et al. 2019](https://doi.org/10.1186/s12916-019-1466-7): the importance of model calibration for decision-making.

## Clinical decision problem

A hospital records the single-lead ECG of all patients admitted to the medical wards. They have a model that provides AF scores for each of these strips. This score in isolation is useless. For any given patient, the team must either act on the AF score and flag them for review by a cardiologist, or ignore it.

These decisions are imperfect. There is a limited number of cardiologists, and time spent reviewing patients is time not spent caring for others, so over-calling patients wastes time on those who do not need it. Under-calling misses potential AF cases and the opportunity to start anticoagulation and prevent strokes. Identifying AF is not necessarily even useful, as many people with AF are managed conservatively once the risk of a clot is weighed against the risk of bleeding. The number of patients you would flag for review therefore depends on the underlying disease, the patients in question, the number of cardiologists you have, and what happens after a patient is flagged.

The ECG data for this problem are the PhysioNet/CinC 2017 single-lead recordings that you've been working with all week. Here, each recording is treated as an individual patient, with its real reference label. The scores from the two models have been simulated for this practical: they stand in for a random forest on features extracted from the ECGs and a 1D ResNet on the raw waveform. Results are given per 1,000 patients.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             roc_curve)

from clinical_utility import check, data, evaluation, plots

scores = data.load_scores()
development = scores[scores["split"] == "development"].reset_index(drop=True)
MODELS = ["random_forest", "resnet"]

print(f"{len(development):,} development patients, {development['af'].mean():.1%} with AF")
development.head()

## Part 1. How much does a policy cost?

The simplest way of thinking about this is in terms of money. A review costs the cardiologist's time, and a missed AF costs the stroke that anticoagulation might have prevented. The cell below puts a rough figure on that second cost from a handful of published numbers ([Hart et al. 2007](https://doi.org/10.7326/0003-4819-146-12-200706190-00007); [Johnson et al. 2016](https://doi.org/10.2147/CEOR.S95662)). You can play around with these values.

In [ ]:
stroke_risk = 0.045      # strokes a year in untreated AF, on average; varies widely between patients
risk_reduction = 0.64    # fall in stroke with anticoagulation (Hart et al., Ann Intern Med 2007)
stroke_cost = 50_000     # assumed: US all-cause spending in the first year after a stroke was $45,000 to $61,000 (Johnson et al., 2016)
share_treated = 0.6      # patients with new AF who would be anticoagulated once bleeding risk is weighed

value_of_finding_af = stroke_risk * risk_reduction * stroke_cost * share_treated
print(f"Illustrative value of finding one AF: about ${value_of_finding_af:,.0f} a year")

This treats a year of spending after a stroke as the cost that anticoagulation would avoid, so it is an illustration rather than an estimate: before the costs of bleeding and of the drugs, finding an AF is worth about \$860 a year. Let's say the cost of missing an AF is \$500 and the cost of a review is \$50, about 10 minutes of a cardiologist's time. These are just round numbers for the exercise and don't reflect any real ward. Feel free to set `share_treated = 0.3` and see how sensitive the result is.

The table below shows the cost of each outcome, and in the next cell these are saved as `COSTS`.

| | Patient has AF | Patient does not have AF |
|---|---|---|
| Flagged for review | True positive (TP): \$50 | False positive (FP): \$50 |
| Not flagged | False negative (FN): \$500 | True negative (TN): \$0 |

In [ ]:
COSTS = {"TP": 50, "FP": 50, "FN": 500, "TN": 0}  # dollars for each patient with this outcome

A **policy** is a decision rule that flags every patient whose score is at or above a **cutoff**. A **flag** here means that the patient's ECG is sent for a cardiologist to review; all other patients are left alone.

### Task 1: calculate outcome counts

Fill in `outcome_counts`, which should return the number of true positives, false positives, false negatives and true negatives. `y` is a boolean array that is True for each patient whose reference label is AF, and `flagged` is a boolean array that is True for each patient the policy sends for cardiologist review. scikit-learn's `confusion_matrix` function returns the four counts, and `.ravel()` can be used to unpack them.

In [ ]:
def outcome_counts(y, flagged):
    """The number of TP, FP, FN and TN, as a pandas Series indexed by those names."""
    return ...


check.outcome_counts(outcome_counts)

### Task 2: calculate the cost of a policy

Complete `policy_cost`, which uses `outcome_counts` to give the cost of a policy in dollars per 1,000 patients.

In [ ]:
def policy_cost(y, flagged, costs):
    """The total cost of these flags, in dollars per 1,000 patients."""
    return ...


check.policy_cost(policy_cost)

This is the policy that `predict()` would give you: the random forest model with a cutoff of 0.5.

In [ ]:
if check.ready(outcome_counts=outcome_counts, policy_cost=policy_cost):
    flagged = development["random_forest"] >= 0.5
    per_1000 = pd.Series(outcome_counts(development["af"], flagged)) * 1000 / len(development)
    display(per_1000.round(1).to_frame("random forest at 0.5, per 1,000 patients"))
    print(f"Cost: ${policy_cost(development['af'], flagged, COSTS):,.0f} per 1,000 patients")

What is the dominant driver of this cost?

**Your answer:** double-click this cell and write your reasoning here.

## Part 2. What is the optimal cutoff?

Every cutoff represents a different policy. The ROC curve visualises this by showing the whole policy space. For every cutoff, it plots what fraction of patients with AF are flagged (true positive rate) against the fraction of those without AF who are flagged (false positive rate).

Below is a schematic diagram of a ROC curve, which is not based on our data but has made-up numbers to demonstrate the point. Each point is a cutoff: the higher the cutoff, the fewer patients are flagged and the closer the point is to the bottom left, and the lower the cutoff, the more patients are flagged and the closer it is to the top right. The dotted diagonal line represents what would be achieved if you flagged patients randomly.

In [ ]:
plots.roc_demo()
plt.show()

### Task 3: evaluate the ROC curve

Complete `roc_points`, which uses scikit-learn's `roc_curve` to produce a DataFrame with the false positive rate and true positive rate for every cutoff. This should include every cutoff, not just the corners of the curve.

In [ ]:
def roc_points(y, score):
    """One row per cutoff, with columns cutoff, fpr and tpr."""
    return ...


check.roc_points(roc_points)

The cell below plots the ROC curve for each model from your `roc_points`. As a sanity check, work out by hand the two rates for the cutoff `CUTOFF`; the cell marks that point, which should sit on the curve. You can then play around with the value of `CUTOFF` to see where other cutoffs sit.

In [ ]:
CUTOFF = 0.5

if check.ready(roc_points=roc_points):
    fig, ax = plots.roc_curves({m: roc_points(development["af"], development[m]) for m in MODELS})
    for model in MODELS:
        flagged = development[model] >= CUTOFF
        fpr = ...  # the share of patients without AF who are flagged
        tpr = ...  # the share of patients with AF who are flagged
        plots.mark(ax, model, fpr, tpr, CUTOFF)
    plt.show()

Where do the curves intersect? Based on this, which model would you select if the ward could only afford a low false positive rate, and which if it could afford a high one? Each model can be summarised by its area under the curve (AUC); however, this doesn't tell us where on the curve to operate.

**Your answer:** double-click this cell and write your reasoning here.

The ROC curve is useful as it captures two of the three rates we care about, though each has a different denominator. The true positive rate (also known as the recall or sensitivity) tells us what fraction of patients who have AF our model will flag. The false positive rate (one minus the specificity) tells us what fraction of patients who do not have AF will be flagged. What neither tells us is what we would observe if we implemented this on the ward: if we flag a patient, what fraction of the time will they have AF? This is the precision (positive predictive value) of the model. Because only about 9% of these patients have AF, even a low false positive rate can mean a large number of false alarms for every correct one. We can see this on the precision-recall curve, which plots the precision against the recall of the model.

In [ ]:
pr = {}
for model in MODELS:
    precision, recall, cutoffs = precision_recall_curve(development["af"], development[model])
    at_half = np.searchsorted(cutoffs, 0.5)  # the first cutoff at or above 0.5
    pr[model] = {"precision": precision, "recall": recall,
                 "average precision": average_precision_score(development["af"], development[model]),
                 "at 0.5": (recall[at_half], precision[at_half])}
plots.pr_curves(pr, prevalence=development["af"].mean()).show()

Which model performs better on this curve? Is this in keeping with the AUC? If you were to use a cutoff of 0.5, what fraction of the patients each model flags would actually have AF?

**Your answer:** double-click this cell and write your reasoning here.

### Task 4: calculate the cost of each cutoff

Complete `price_cutoffs`, which uses `roc_points` to determine the cost at each cutoff. It should convert the false and true positive rates to numbers of patients, then to flags, AF found, AF missed and the cost, each per 1,000 patients.

In [ ]:
def price_cutoffs(y, score, costs):
    """One row per cutoff, with columns cutoff, flags, af_found, af_missed and cost, per 1,000 patients."""
    return ...


check.price_cutoffs(price_cutoffs)

The cheapest cutoff for each model, and the cost of every cutoff, are shown below:

In [ ]:
if check.ready(price_cutoffs=price_cutoffs):
    tables = {model: price_cutoffs(development["af"], development[model], COSTS) for model in MODELS}
    cheapest = pd.DataFrame({model: table.loc[table["cost"].idxmin()]
                             for model, table in tables.items()}).T
    display(cheapest.round(3))
    plots.costs_by_flags(tables).show()

Which model is the most cost-effective at these prices? How many more patients does it flag than the other model? How far from 0.5 is each model's cheapest cutoff?

**Your answer:** double-click this cell and write your reasoning here.

## Part 3. Is the money necessary?

These dollar figures are coarse estimates, and the cost of each outcome is not easy to determine. For example, the value of detecting AF differs between patients: for an 85-year-old who has had a stroke it is much higher than for a 40-year-old with no risk factors, for whom anticoagulation may do more harm than good. Similarly, the cost of a review is not fixed. If a cardiologist has free time it is small, but if they must be pulled away from other duties it is much larger. The cost of a stroke to a hospital is not the same as its cost to the patient and their family, and much of that cost is disability and lost quality of life, which have no agreed value. There is no consensus on these values, so they will differ from person to person. What happens if we double all the costs?

In [ ]:
if check.ready(price_cutoffs=price_cutoffs):
    doubled = {outcome: 2 * cost for outcome, cost in COSTS.items()}
    for model in MODELS:
        again = price_cutoffs(development["af"], development[model], doubled)
        print(f"{model}: cheapest cutoff {cheapest.loc[model, 'cutoff']:.3f} at our costs, "
              f"{again.loc[again['cost'].idxmin(), 'cutoff']:.3f} at double")

We get the same cheapest cutoffs. The absolute costs don't matter; what matters is the ratio between them. We can see why by considering a single patient whose probability of AF is $p$. We would flag them if the expected cost of flagging them is no more than the expected cost of ignoring them:

$$p \cdot C_{TP} + (1 - p) \cdot C_{FP} \le p \cdot C_{FN} + (1 - p) \cdot C_{TN}.$$

### Task 5: threshold probability

Complete `threshold_probability`, which solves that inequality for $p$: the minimum probability of AF that would justify flagging the patient.

In [ ]:
def threshold_probability(costs):
    """The probability of AF above which flagging a patient costs less than ignoring them."""
    return ...


check.threshold_probability(threshold_probability)
if check.ready(threshold_probability=threshold_probability):
    p_t = threshold_probability(COSTS)
    print(f"Flag a patient when P(AF) is at least {p_t:.2f}: up to {(1 - p_t) / p_t:.0f} "
          "unnecessary reviews for each AF found.")

In reality, you don't need a dollar value to find $p_t$; you can simply ask the people who will act on the flags. One way of phrasing it is to ask how many patients without AF they would be willing to review to find a single patient with AF. If they say they would be willing to review 9 patients without AF for every patient with AF, then their threshold probability is $p_t = 1 / (1 + 9) = 0.1$. You could similarly ask, 'How sure would you have to be that a patient has AF to send them to a cardiologist?' An answer of 'about one in ten' would again be a threshold probability of 0.1. This threshold varies depending on who you ask, which ward you're in and the type of patients you're seeing, so it is important to compare the models across a range of threshold probabilities rather than at a single value.

## Part 4. Decision curve analysis

Decision curve analysis ([Vickers and Elkin, 2006](https://doi.org/10.1177/0272989X06295361)) is a method of comparing models that does not need a price for each outcome, only the threshold probability. The threshold probability is the point at which a clinician would be indifferent between flagging and not flagging a patient: the value of finding one more AF is the same as $(1 - p_t) / p_t$ unnecessary reviews. We can invert this to ask what fraction of an AF found one unnecessary review costs: $p_t / (1 - p_t)$. If $p_t = 0.1$, a false positive costs a ninth of an AF found; if $p_t = 0.2$, a quarter. We can use this to weigh up the false positives a model produces against its true positives, which gives the net benefit of the model:

$$\text{net benefit} = \frac{TP}{n} - \frac{FP}{n} \cdot \frac{p_t}{1 - p_t}.$$

A net benefit of 0.05 can be thought of as detecting 5 patients with AF per 100 patients monitored, with no incorrect flags.

### Task 6: net benefit

Complete `net_benefit`, which uses `outcome_counts` to calculate the net benefit of a policy at threshold probability $p_t$.

In [ ]:
def net_benefit(y, flagged, p_t):
    """Net benefit of these flags at threshold probability p_t, per patient."""
    return ...


check.net_benefit(net_benefit)

The first point to consider is what would happen if we did not have a model. We would have two choices that could be made blind: flag everyone, or flag no one. If we flagged no one, we would expect a net benefit of 0, as we would detect no true positives and have no false positives. If we flagged everyone, we would detect all the true positives but also flag every patient without AF, so we would only have a positive net benefit at very low threshold probabilities, where a false positive carries little penalty. A model would need to outperform both of these choices at the thresholds that matter for the ward, as well as any other simple rule that could be used instead. For example, a simple rule may be to flag every patient over the age of 60. A decision curve shows this by plotting the net benefit of flagging every patient at or above each threshold risk, alongside the two blind choices.

In [ ]:
THRESHOLDS = np.round(np.arange(0.02, 0.41, 0.01), 2)


def decision_curve(y, model_scores, thresholds):
    """Net benefit of each model, of flagging everyone and of flagging no one, at each threshold."""
    curves = {name: [net_benefit(y, score >= t, t) for t in thresholds]
              for name, score in model_scores.items()}
    curves["flag all"] = [net_benefit(y, np.ones(len(y), dtype=bool), t) for t in thresholds]
    curves["flag none"] = [0.0] * len(thresholds)
    return pd.DataFrame(curves, index=thresholds)


if check.ready(net_benefit=net_benefit):
    curves = decision_curve(development["af"], {m: development[m] for m in MODELS}, THRESHOLDS)
    plots.decision_curves(curves).show()
    display(curves.loc[[0.05, 0.1, 0.2]].round(4))

At your threshold probability from Task 5, which model is superior according to the decision curve? Is this the same model you found to be cheaper in Part 2 at the same costs? At what threshold probabilities is each model superior to the two uninformed choices?

**Your answer:** double-click this cell and write your reasoning here.

## Part 5. Why the two answers are different

In Part 2, the way we found the most cost-effective policy was simply to try every cutoff, so we did not need to worry about what the model's output meant. When drawing the decision curve, however, we use the model's output itself to decide which patients are flagged. This means the model's output must mean what it says: patients given a score of 0.1 should have a 10% chance of having AF. This is referred to as the calibration of the model.

A model is perfectly calibrated if, when we take all the patients given a score of 0.1, about 10% of them have AF. This is not the same as discrimination, which measures how well the model separates the two classes: a model can perform well on the ROC curve and still be poorly calibrated. This typically happens with very flexible models trained purely to separate the classes, which tend to be overconfident, with scores pushed towards zero and one. It matters whenever the model's output is used directly for a decision, such as in a decision curve, with a fixed threshold, or when a clinician reads a score as a risk. A model's calibration can be seen on a calibration curve, which plots the share of patients with AF in each bin of scores against the bin's average score; a calibrated model sits on the diagonal.

In [ ]:
plots.calibration({model: calibration_curve(development["af"], development[model], n_bins=10,
                                            strategy="quantile")
                   for model in MODELS}).show()

We can improve a model's calibration by learning a mapping that transforms its output; this is known as recalibration. The simplest method is to fit a logistic regression of the true label on the logit of the model's output, $\log(s / (1 - s))$. This is known as Platt scaling, and it maps the model's output to an estimate of the probability that the patient has AF. When the fitted slope is positive, as it is here, the map only ever increases, so it doesn't change the order of the patients, and the ROC curve and the AUC stay the same. The map can only be fitted on data that was not used in training, or in any other way that would leak information; in our case we can use the calibration patients.

### Task 7: recalibrate a risk score

Fill in the `recalibrate` function to perform Platt scaling. It should fit the calibration model to the predictions `fit_score` and the true values `fit_y`, and then produce predictions for `score`. This can be done with scipy's `logit` function and scikit-learn's `LogisticRegression` with no regularisation (`C=np.inf`). Don't forget that scikit-learn expects a 2D feature array, so for a single feature the array will need to be reshaped with `.reshape(-1, 1)`, or the column selected as a one-column DataFrame.

In [ ]:
def recalibrate(fit_score, fit_y, score):
    """Probabilities for score, from a Platt map fitted on fit_score and fit_y."""
    return ...


check.recalibrate(recalibrate)

Now that we can recalibrate a model, let's apply it to the ResNet, fitted on the calibration patients, and see what it does to the calibration curve and the decision curve.

In [ ]:
calibration_patients = scores[scores["split"] == "calibration"]
if check.ready(recalibrate=recalibrate, net_benefit=net_benefit):
    development["resnet_recalibrated"] = recalibrate(
        calibration_patients["resnet"], calibration_patients["af"], development["resnet"])
    candidates = ["random_forest", "resnet", "resnet_recalibrated"]
    plots.calibration({model: calibration_curve(development["af"], development[model], n_bins=10,
                                                strategy="quantile")
                       for model in candidates}).show()
    curves = decision_curve(development["af"], {m: development[m] for m in candidates}, THRESHOLDS)
    plots.decision_curves(curves).show()
    display(curves.loc[[0.05, 0.1, 0.2]].round(4))

Does the recalibrated ResNet now agree with Part 2? For which range of threshold probabilities would you use the ResNet, and for which the random forest?

**Your answer:** double-click this cell and write your reasoning here.

## Part 6. What would you recommend?

### Task 8: policy recommendation

Please write a short recommendation for the ward: which model to use (`"random_forest"`, `"resnet"` or `"resnet_recalibrated"`), at what threshold probability to flag patients, why, and what would make you revisit the decision once it is running. Your demonstrator will review your response.

In [ ]:
recommendation = {
    "model": ...,
    "threshold": ...,
    "reasoning": ...,    # why this model and threshold, in a sentence or two
    "reopen_if": ...,    # what you would monitor once it runs, and what would make you look again
}
check.recommendation(recommendation)

The patients in the next cell were not involved in any of your decisions. Once you've seen the results, the test set has been used: if you change your recommendation and run this again, you are no longer performing a test.

In [ ]:
if check.ready(recommendation=recommendation, outcome_counts=outcome_counts, policy_cost=policy_cost,
               recalibrate=recalibrate):
    held_out = scores[scores["split"] == "evaluation"].copy()
    held_out["resnet_recalibrated"] = recalibrate(
        calibration_patients["resnet"], calibration_patients["af"], held_out["resnet"])
    flagged = held_out[recommendation["model"]] >= recommendation["threshold"]
    display(evaluation.report(held_out["af"], flagged, outcome_counts).round(1))
    print(f"Cost: ${policy_cost(held_out['af'], flagged, COSTS):,.0f} per 1,000 patients")

## Extension: review limits

This section is an optional extension to the practical, if you have time.

Up until this point we've assumed that we could review as many patients as we wanted; in reality, there is a limited number of cardiologists to review them. Let's consider a case where the wards produce 300 strips a day and the cardiologists can perform 30 reviews a day, about five hours of their time at 10 minutes a review. On average this is 100 reviews per 1,000 patients, but the number flagged fluctuates from day to day, so on a particularly busy day flagged patients would have to wait for a review.

For the following extension tasks, we will use a `price_cutoffs` table like the ones in `tables` above: a DataFrame with one row per cutoff and the columns `cutoff`, `flags`, `af_found`, `af_missed` and `cost`, each per 1,000 patients.

### Extension task A: most cost-effective policy under the average limit

Fill in `cheapest_within_capacity` to return the row of `table` with the lowest cost among the policies that flag at most `max_flags` patients per 1,000.

In [ ]:
def cheapest_within_capacity(table, max_flags):
    """The cheapest row of a price_cutoffs table that flags at most max_flags per 1,000."""
    return ...


check.cheapest_within_capacity(cheapest_within_capacity)
STRIPS_PER_DAY, REVIEWS_PER_DAY = 300, 30
if check.ready(cheapest_within_capacity=cheapest_within_capacity, price_cutoffs=price_cutoffs):
    average_limit = 1000 * REVIEWS_PER_DAY / STRIPS_PER_DAY
    within_average = pd.DataFrame({model: cheapest_within_capacity(table, average_limit)
                                   for model, table in tables.items()}).T
    display(within_average.round(3))

### Extension task B: policy overflow frequency

A policy that flags 100 out of every 1,000 on average will only flag exactly 30 a day on average. If we assume that exactly `strips_per_day` strips are received a day, and that each one has the same independent probability `flags_per_1000 / 1000` of being flagged, the number of flags per day follows a binomial distribution. Fill in `share_of_days_over` to work out on what fraction of days the policy will flag more patients than the cardiologists are able to review. This can be simulated with numpy's random generator (`np.random.default_rng(seed)`) over `days` days, returning the fraction of days that are over capacity.

In [ ]:
def share_of_days_over(flags_per_1000, strips_per_day, reviews_per_day, days=10_000, seed=0):
    """The share of simulated days on which the flags exceed reviews_per_day."""
    return ...


check.share_of_days_over(share_of_days_over)
if check.ready(share_of_days_over=share_of_days_over, cheapest_within_capacity=cheapest_within_capacity,
               price_cutoffs=price_cutoffs):
    display(pd.Series({model: share_of_days_over(row["flags"], STRIPS_PER_DAY, REVIEWS_PER_DAY)
                       for model, row in within_average.iterrows()}).round(3)
            .to_frame("share of days over capacity, at the average limit"))

### Extension task C: most cost-effective policy with rare overflow

Fill in `cheapest_reliable` to find, using `share_of_days_over`, the cheapest row of `table` that overflows on at most `max_share` of days.

In [ ]:
def cheapest_reliable(table, strips_per_day, reviews_per_day, max_share=0.05):
    """The cheapest row of a price_cutoffs table that overflows on at most max_share of days."""
    return ...


check.cheapest_reliable(cheapest_reliable)
if check.ready(cheapest_reliable=cheapest_reliable, cheapest_within_capacity=cheapest_within_capacity,
               price_cutoffs=price_cutoffs):
    reliable = pd.DataFrame({model: cheapest_reliable(table, STRIPS_PER_DAY, REVIEWS_PER_DAY)
                             for model, table in tables.items()}).T
    display(reliable.round(3))
    rng = np.random.default_rng(0)
    plots.daily_flags({f"{model}, {label}": rng.binomial(STRIPS_PER_DAY, row["flags"] / 1000, size=365)
                       for label, frame in (("average limit", within_average), ("rarely over", reliable))
                       for model, row in frame.iterrows()}, REVIEWS_PER_DAY).show()

The plot shows a year of simulated days for each policy, with a line marking the cardiologists' daily capacity. Finally, the cell below shows how the cost per 1,000 patients of the cheapest policy within the average limit changes as the cardiologists' capacity increases:

In [ ]:
if check.ready(cheapest_within_capacity=cheapest_within_capacity, price_cutoffs=price_cutoffs):
    limits = range(20, 401, 10)
    by_limit = pd.DataFrame({model: [cheapest_within_capacity(table, limit)["cost"] for limit in limits]
                             for model, table in tables.items()}, index=limits)
    plots.capacity_curves(by_limit).show()

How much more expensive is it per 1,000 patients to use a policy that overflows on only 5% of days instead of one that meets the average? How many reviews per 1,000 patients would the cardiologists have to be able to perform to make the ResNet the better choice? What would you tell the hospital about staffing if it wanted to deploy the ResNet?

**Your answer:** double-click this cell and write your reasoning here.

## Your progress

Please run this cell and leave the notebook open for your demonstrator to view. We will conclude the session with a discussion on what would be required to implement this on a real-world ward.

In [ ]:
check.progress(globals())